<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Las LSTM Gates</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Las LSTM introducen puertas para controlar qué información se conserva, qué se olvida y qué se incorpora al estado interno. Este notebook se centra en esa solución a las limitaciones de memoria de las RNN más simples.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Las puertas permiten manejar dependencias más largas y mejorar la estabilidad del aprendizaje secuencial.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender por qué existen las LSTM y qué función cumple cada gate dentro del flujo de información.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Más complejidad en la celda no significa complicación gratuita; responde a un problema real de memoria y gradientes.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Una LSTM regula explícitamente el paso de información mediante mecanismos de olvido, entrada y salida.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación de una secuencia sintética
"""Este bloque crea un tensor que representa una secuencia.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "secuencia" contiene un tensor que representa una secuencia con 4 pasos temporales, cada uno con una característica de entrada.
Aplicamos el método ".unsqueeze()" a la variable "secuencia" para agregar una dimensión adicional al tensor.
El argumento "0" indica que la nueva dimensión se agregará en la posición 0 del tensor.
La variable "secuencia" contiene valores ordenados que serán procesados por la LSTM.
El formato batch-first permite integrarla directamente en la arquitectura recurrente."""

import torch

secuencia = torch.tensor([[0.2],
                          [0.4],
                          [0.6],
                          [0.8]])

secuencia = secuencia.unsqueeze(0)

secuencia # Muestra el tensor con la dimensión adicional agregada.


In [ ]:
# Bloque 2 — Definición de una LSTM simple
"""Este bloque define una red LSTM utilizando nn.LSTM.
Este bloque importa "torch.nn" para definir capas de redes neuronales recurrentes en PyTorch.

Aplicamos la clase "nn.LSTM()" a la variable "lstm" para definir una red LSTM simple.
El parámetro "input_size=" indica el número de características de entrada en cada paso de la secuencia.
El parámetro "hidden_size=" define el tamaño de los estados oculto y de celda de la LSTM.
El parámetro "num_layers=" especifica el número de capas recurrentes apiladas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada y de la salida representa el lote ("True" en este caso).

La variable "lstm" contiene una instancia de la clase "nn.LSTM()", que representa la red LSTM definida.
La red procesa cada elemento de la secuencia y actualiza tanto el estado oculto como el estado de celda.
El tamaño del estado oculto y de la celda influye en la capacidad de la red para representar información temporal."""

import torch.nn as nn

lstm = nn.LSTM(input_size=1, hidden_size=3, num_layers=1, batch_first=True)

lstm  # Muestra la instancia de la red LSTM definida.


In [ ]:
# Bloque 3 — Inicialización de estados oculto y de celda
"""Este bloque inicializa el estado oculto y el estado de celda con valores cero.
Aplicamos la función "torch.zeros()" a la variable "estado_oculto" para crear un tensor de ceros que representa el estado oculto inicial de la red.
El primer argumento "1" indica el número de capas recurrentes apiladas (en este caso, 1 capa).
El segundo argumento "1" indica el tamaño del lote de entrada (en este caso, una secuencia por lote).
El tercer argumento "3" indica el tamaño del estado oculto de la LSTM (el número de unidades en la capa oculta).
La variable "estado_oculto" representa el estado oculto inicial de la LSTM.

Aplicamos la función "torch.zeros()" a la variable "estado_celda" para crear un tensor de ceros que representa el estado de celda inicial de la red.
El primer argumento "1" indica el número de capas recurrentes apiladas (en este caso, 1 capa).
El segundo argumento "1" indica el tamaño del lote de entrada (en este caso, una secuencia por lote).
El tercer argumento "3" indica el tamaño del estado de celda de la LSTM.
La variable "estado_celda" representa el estado de celda inicial, cuya información regulan las compuertas."""

estado_oculto = torch.zeros(1, 1, 3)

estado_celda = torch.zeros(1, 1, 3)

estado_oculto, estado_celda # Muestra los estados iniciales de la LSTM: el estado oculto y el estado de celda.


In [ ]:
# Bloque 4 — Forward pass de la LSTM
"""Este bloque ejecuta el forward pass de la LSTM.
Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase a "salida", "h_n" y "c_n" mediante la expresión "lstm(secuencia, (estado_oculto, estado_celda))".
El argumento "secuencia" es el tensor de entrada que contiene los pasos temporales de la secuencia.
El argumento "(estado_oculto, estado_celda)" es el estado inicial de la LSTM, que incluye tanto el estado oculto como el estado de celda.

La variable "salida" contiene las salidas generadas en cada paso temporal.
Las variables "h_n" y "c_n" contienen el estado oculto final y el estado de celda final, respectivamente.
En esta LSTM de una capa y una dirección, la última salida coincide con el estado oculto final."""

salida, (h_n, c_n) = lstm(secuencia, (estado_oculto, estado_celda))

salida, (h_n, c_n) # Muestra la salida de la LSTM, el estado oculto final y el estado de celda final.


In [ ]:
# Bloque 5 — Inspección de la evolución temporal
"""Este bloque extrae las salidas generadas en cada paso temporal.
Aplicamos los métodos "squeeze()" y "tolist()" a la variable "salida" para obtener la evolución temporal de las salidas de la LSTM.
El método "squeeze()" elimina la dimensión del lote, cuyo tamaño es uno.
El argumento "0" indica que se eliminará la primera dimensión del tensor, que corresponde al lote.
El método "tolist()" convierte el tensor resultante en una lista de Python, facilitando su inspección.
La variable "evolucion" almacena las representaciones producidas por la LSTM.
La operación permite observar las salidas paso a paso, pero no muestra directamente los valores de las compuertas."""

evolucion = salida.squeeze(0).tolist()

evolucion # Muestra la evolución temporal de las salidas de la LSTM.


In [ ]:
# Bloque 6 — Resultados finales
"""Este bloque organiza los resultados del procesamiento con LSTM.
La variable "resultado" almacena en forma de diccionario la secuencia original, las salidas temporales y los estados finales como listas.
La estructura permite observar las salidas y los estados finales, pero no los valores individuales de las compuertas."""

resultado = {
    "secuencia": secuencia.squeeze(0).tolist(),      # El método ".squeeze(0)" elimina la dimensión de lote, de tamaño 1.
    "salidas_temporales": evolucion,
    "estado_oculto_final": h_n.squeeze(0).tolist(),  # El método ".squeeze(0)" elimina la dimensión de capa, de tamaño 1.
    "estado_celda_final": c_n.squeeze(0).tolist()    # El método ".squeeze(0)" elimina la dimensión de capa, de tamaño 1.
}

resultado # Muestra el diccionario con la secuencia original, las salidas temporales, el estado oculto final y el estado de celda final.



<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo las LSTM intentan gestionar mejor la memoria interna de una red secuencial.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Las gates existen para decidir qué información merece persistir y cuál conviene descartar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Memorizar los nombres de las puertas sin relacionarlas con el problema práctico de retener contexto útil.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Transfer Learning para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Transfer Learning y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">11_Transfer_learning/01_El_transfer_learning.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">